In [ ]:
import sys
from pathlib import Path
import yaml
import torch

root_dir = Path.cwd().parents[2]
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

from src.pretraitement.pretretement_data import pretretement
from src.espace.detection_from_scratch.model import SimpleYOLO

In [ ]:
yaml_path = Path("data.yaml")
with open(yaml_path, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

NUM_CLASSES = data_cfg["nc"]
CLASSES_NAMES = data_cfg["names"]
IMG_SIZE = 640

print("Classes count:", NUM_CLASSES)

In [ ]:
train_loader, val_loader, test_loader = pretretement(batch_size=8, img_size=IMG_SIZE)

print('Train samples:', len(train_loader.dataset))
print('Validation samples:', len(val_loader.dataset))
print('Test samples:', len(test_loader.dataset))

In [ ]:
images, targets = next(iter(train_loader))
print("Batch images shape:", images.shape)
print("Targets list length:", len(targets))
if len(targets) > 0:
    print("First sample targets shape:", targets[0].shape)

In [ ]:
model = SimpleYOLO(num_classes=NUM_CLASSES)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
out_s, out_m, out_l = model(images)
print("Output Small:", out_s.shape)
print("Output Medium:", out_m.shape)
print("Output Large:", out_l.shape)

In [ ]:
from src.espace.detection_from_scratch.model import train_model

train_model(model, train_loader, val_loader, epochs=5, lr=1e-3)